# 08_10_4_つの解かせ方で_正解と出力トークンを測る

コード：`ch08/blocks/08_10_4_つの解かせ方で_正解と出力トークンを測る.ipynb`

本文の断片は参照表示し、末尾のセルで前提を補う既存ランナーを実行します。関数・変数・外部処理が省略されているため、断片だけを単独実行しないでください。

モデル・データの取得、または長い計算を含みます。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import os
import pathlib
import sys
import zipfile

CODE = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents,
                        pathlib.Path.cwd() / 'gradient-attention-agent']
             if (p / 'run_all.py').is_file() and (p / 'common/blocks.py').is_file()), None)
if CODE is None:
    from google.colab import files
    uploaded = files.upload()  # gradient-attention-agent-code.zip を選ぶ
    if len(uploaded) != 1:
        raise ValueError('配布ZIPを1つ選んでください')
    dest = pathlib.Path.cwd().resolve()
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        for member in z.namelist():
            if not (dest / member).resolve().is_relative_to(dest / 'gradient-attention-agent'):
                raise ValueError('同梱のコード配布ZIPを選んでください')
        z.extractall(dest)
    CODE = dest / 'gradient-attention-agent'
    del uploaded
print('コード一式:', CODE)


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2', 'transformers==5.22.1', 'tiktoken==0.14.0', 'scikit-learn>=1.5,<2', 'pydantic==2.14.2', 'outlines==1.3.3', 'lancedb==0.41.0', 'pyarrow==26.0.1', 'sentence-transformers==6.2.0']], check=True)


## 本文の掲載コード（参照）


```python
# 8 章 12 節：直接・例つき CoT・「段階的に考えて」・多数決（Self-Consistency）を、算数の文章題で比べる
import re, json, sys, collections, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 本書で用意した文章題（2〜3 段の計算が要る）と正解
PROBLEMS = [
    ("りんごが 12 個ある。3 人で同じ数ずつ分けたあと、1 人がさらに 2 個もらった。その人のりんごは何個か。", 6),
    ("1 冊 350 円のノートを 4 冊買い、2000 円札で払った。おつりは何円か。", 600),
    ("時速 60 km で 2 時間走り、そのあと時速 40 km で 30 分走った。全部で何 km 走ったか。", 140),
    ("教室に 28 人いる。男子は女子より 4 人多い。女子は何人か。", 12),
    ("1 箱 6 個入りのたまごを 5 箱買った。料理に 13 個使った。残りは何個か。", 17),
    ("長さ 3 m のひもから 45 cm のひもを 6 本切り取った。残りは何 cm か。", 30),
    ("ある数に 7 を足して 3 倍すると 45 になる。ある数はいくつか。", 8),
    ("1 時間に 15 ページ読む人が、240 ページの本を読み終えるには何時間かかるか。", 16),
    ("1 個 120 円のパンを 3 個と、1 本 150 円のジュースを 2 本買った。合計は何円か。", 660),
    ("貯金が 5000 円ある。毎週 300 円ずつ使うと、何週間後に残りが 2000 円になるか。", 10),
    ("縦 8 cm、横 5 cm の長方形が 3 枚ある。面積の合計は何平方 cm か。", 120),
    ("A さんは 14 歳、お父さんは 42 歳。お父さんの年齢が A さんの 2 倍になるのは何年後か。", 14),
    ("1 ダースの鉛筆を 4 ダース買い、クラスの 30 人に 1 本ずつ配った。残りは何本か。", 18),
    ("定価 2400 円の品物が 25% 引きで売られている。売り値は何円か。", 1800),
    ("バスに 23 人乗っていた。最初の停留所で 8 人降りて 5 人乗り、次の停留所で 6 人降りた。今何人乗っているか。", 14),
    ("1 分間に 4 L 水が出る蛇口で、90 L の水そうをいっぱいにする。すでに 18 L 入っているとき、あと何分かかるか。", 18),
    ("3 つの数の平均が 20 で、そのうち 2 つが 15 と 22 である。残りの数はいくつか。", 23),
    ("1 本 80 円の鉛筆と 1 個 50 円の消しゴムを、それぞれ 6 つずつ買った。1000 円払うとおつりは何円か。", 220),
    ("ある本の 3 分の 1 を 1 日目に、残りの半分を 2 日目に読んだ。残りは 40 ページだった。本は何ページか。", 120),
    ("姉は妹より 500 円多く持っていて、2 人の合計は 3100 円である。妹はいくら持っているか。", 1300),
]
EXAMPLES = ("問題：みかんが 20 個ある。4 人で同じ数ずつ分け、1 人が 1 個食べた。その人のみかんは何個か。\n"
            "考え方：20 ÷ 4 = 5 個ずつ。1 個食べたので 5 − 1 = 4。\n答え：4\n\n"
            "問題：1 個 200 円のケーキを 3 個買い、1000 円払った。おつりは何円か。\n"
            "考え方：200 × 3 = 600 円。1000 − 600 = 400。\n答え：400\n\n")
TAIL = "最後の行に「答え：数」と、数字だけを書く。"
CONDS = {
    "① 直接答えさせる":            lambda q: f"次の問題の答えの数だけを「答え：数」の形で書く。\n問題：{q}",
    "② 例つきの CoT（例 2 つ）":   lambda q: f"次の例にならって、考え方を書いてから答える。{TAIL}\n\n{EXAMPLES}問題：{q}",
    "③ 「段階的に考えて」と付ける": lambda q: f"次の問題を、段階的に考えてから答える。{TAIL}\n問題：{q}",
}

name = sys.argv[1] if len(sys.argv) > 1 else "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name).eval()

@torch.no_grad()
def ask(prompt, sample=False, n=1, seed=0):
    ids = tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)
    torch.manual_seed(seed)
    out = model.generate(**ids, max_new_tokens=256, do_sample=sample, temperature=0.7 if sample else None,
                         top_p=0.95 if sample else None, num_return_sequences=n)
    gen = out[:, ids["input_ids"].shape[1]:]
    texts = [tok.decode(g, skip_special_tokens=True) for g in gen]
    lens = [int((g != tok.pad_token_id).sum()) for g in gen]
    return texts, lens

def parse(text):                       # 最後の「答え：数」を読む（無ければ最後の数）
    m = re.findall(r"答え[：:]\s*(-?[\d,]+)", text) or re.findall(r"-?\d[\d,]*", text)
    return int(m[-1].replace(",", "")) if m else None

rows, raw = {}, []
for k, make in CONDS.items():
    ok, toks = 0, 0
    for q, a in PROBLEMS:
        (t,), (L,) = ask(make(q))
        ok += parse(t) == a; toks += L
        raw.append({"cond": k, "q": q, "gold": a, "answer": t})
    rows[k] = (ok, toks / len(PROBLEMS))
ok, toks, single = 0, 0, 0
for i, (q, a) in enumerate(PROBLEMS):                      # ④ ③を 5 本引いて多数決
    texts, lens = ask(CONDS["③ 「段階的に考えて」と付ける"](q), sample=True, n=5, seed=i)
    votes = collections.Counter(p for p in map(parse, texts) if p is not None)
    single += sum(parse(t) == a for t in texts)            # 引いた 1 本ずつの正解（多数決の前）
    ok += bool(votes) and votes.most_common(1)[0][0] == a; toks += sum(lens)
    raw.append({"cond": "④", "q": q, "gold": a, "answers": texts})
rows["④ ③を 5 本引いて多数決"] = (ok, toks / len(PROBLEMS))
print(f"モデル：{name}")
for k, (ok, t) in rows.items():
    print(f"{k}：正解 {ok}/20  出力トークン（1 問あたり） {t:.0f}")
print(f"（④で温度 0.7 で引いた 1 本ずつの正解：{single}/100）")
json.dump(raw, open(f"cot_{name.split('/')[-1]}_raw.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)   # 証跡：全件の応答
```


## 前提を補って実行する

既存ランナーが関連する断片と必要なデータ・スタブを接続します。外部サービスを実行した証明や、掲載用AI応答の完全証跡にはなりません。


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, str(CODE / "run_all.py"), "--only", "08", "--name=cot_0.5B", "--skip-docker"], cwd=CODE, check=True)
